# 🌊 `animgen` Quickstart: Procedural Rigging & Biomechanical Animation

Welcome to the **`animgen`** quickstart tutorial!

**`animgen`** is a standalone, procedural rigging, discrete heat skinning, and biomechanical animation synthesis framework built natively in Python over **glTF 2.0 / PyGLTFLib**. It transforms arbitrary static 3D meshes into articulated, skinned, and procedurally animated assets with continuous wave kinematics—**with zero runtime dependency on Blender**.

### In this notebook, you will learn how to:
1. **Autorig & Animate Fish (`FishModels`)**: Load a static fish mesh, automatically segment appendages, extract spine splines, solve Laplacian heat skin weights, and synthesize biological multi-speed swimming clips (`swim`, `idle`, `sprint`).
2. **Autorig & Animate Serpentine Organisms (`SerpentineModels`)**: Synthesize continuous Anguilliform travelling waves along high-density articulated spines for snakes and eels.
3. **Synthesize Custom Motion Clips via `generate_base_animation`**: Use `FishModels.generate_base_animation` to synthesize custom movement kinematics (e.g., high-speed escape bursts or gentle glides) and attach them directly to the model's animator.
4. **Inspect Armature & Animation Data**: Inspect bone hierarchies, DAG joints, skin weights, and keyframed channels.

In [ ]:
import sys
from pathlib import Path

# Resolve project root
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

from animgen import AnimationClip, BaseModelClass, FishModels, SerpentineModels  # noqa: E402

# Ensure output directory exists
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Outputs directory: {output_dir}")

---
## 1. Fish Autorigging & Swimming Locomotion (`FishModels`)

In just a few lines of Python, `FishModels` performs:
- **Appendage Segmentation**: Isolates dorsal, caudal, and pectoral fins using 3D Shape Diameter Function (SDF) and graph analysis.
- **Medial Skeleton Extraction**: Computes cotangent Laplace-Beltrami contraction and slice centroid refinement to produce a centripetal Catmull-Rom spine.
- **Discrete Laplacian Heat Skinning**: Solves per-vertex bone weights in $< 20\text{ ms}$.
- **Procedural Locomotion Synthesis**: Generates multi-speed swimming clips (`swim`, `idle`, `sprint`) with natural tail steering and synchronized pectoral flapping.

> **Note on SAM3 Vision Weights:** This quickstart sets `use_sam=False` to execute rapidly on any CPU via pure 3D SDF geometry. To enable Meta SAM3 neural vision segmentation, copy `.env.example` to `.env`, set `HF_TOKEN`, and pass `use_sam=True`.


In [ ]:
# 1. Load static fish model from generated_data
goldfish_path = project_root / "generated_data" / "models" / "dec_mesh_Goldfish.glb"
model = BaseModelClass(str(goldfish_path))
print(
    f"Loaded mesh: {len(model.mesh.vertices)} vertices, {len(model.mesh.faces)} faces"
)

# 2. Initialize and execute end-to-end autorigging & swimming animation
# (use_sam=False runs fast CPU geometric SDF segmentation without requiring SAM3 vision weights)
pipeline = FishModels(model, use_sam=False)
animated_fish = pipeline.process()

# 3. Export engine-ready animated glTF 2.0 binary with swim, idle, sprint tracks
fish_output = output_dir / "quickstart_goldfish_swimming.glb"
pipeline.export(str(fish_output))

print(f"[+] Successfully exported animated fish to: {fish_output}")
print(
    f"[+] Registered animation tracks: {[clip.name for clip in pipeline.model.animator.clips]}"
)

---
## 2. Serpentine / Snake Autorigging & Wave Locomotion (`SerpentineModels`)

For elongated species like sea snakes, eels, and lampreys, `SerpentineModels` constructs continuous Anguilliform travelling waves along high-density articulated spines.

In [ ]:
# 1. Load static snake model
snake_path = project_root / "generated_data" / "models" / "dec_mesh_Sea_Snake.glb"
snake_model = BaseModelClass(str(snake_path))
print(
    f"Loaded snake mesh: {len(snake_model.mesh.vertices)} vertices, {len(snake_model.mesh.faces)} faces"
)

# 2. Run serpentine autorigging with 20-bone spine & continuous travelling waves
snake_pipeline = SerpentineModels(snake_model, num_bones=20)
snake_pipeline.process()

# 3. Export with 'slow' and 'fast' locomotion animations
snake_output = output_dir / "quickstart_snake_locomotion.glb"
snake_pipeline.export(str(snake_output))

print(f"[+] Successfully exported serpentine model to: {snake_output}")
print(
    f"[+] Registered animation tracks: {[clip.name for clip in snake_pipeline.model.animator.clips]}"
)

---
## 3. Adding Custom Motion Clips via `generate_base_animation`

`FishModels.generate_base_animation` provides direct programmatic access to the kinematic wave synthesis engine. You can customize:
- `wave_amplitude`: Peak undulation excursion
- `wave_duration`: Cycle period in seconds (e.g. 0.6s for high-frequency burst vs 3.5s for idle)
- `pectoral_mode`: `'active'` (flapping & feathering) vs `'closed'` (tucked against body)
- `pectoral_flap_deg` & `pectoral_pitch_deg`: Excursion angles
- `dorsal_flex_deg`: Dorsal fin stabilization angle

Once generated, wrap the keyframe rotation matrices into an `AnimationClip` and register it on `pipeline.model.animator.add_animation_clip(clip)`.

In [ ]:
# 1. Synthesize custom kinematic keyframes for a high-speed escape burst
burst_positions = FishModels.generate_base_animation(
    armature=pipeline.model.armature,
    wave_amplitude=0.35,
    wave_duration=0.6,
    tail_orientation=pipeline.tail_orientation,
    pectoral_mode="active",
    pectoral_flap_deg=20.0,
    pectoral_pitch_deg=8.0,
    dorsal_flex_deg=5.0,
)

print(f"Synthesized {len(burst_positions)} keyframes for 'escape_burst' clip.")

# 2. Wrap into an AnimationClip and register on the model's animator
burst_clip = AnimationClip(
    name="escape_burst",
    duration=0.6,
    armature=pipeline.model.armature,
    is_loopable=True,
)
burst_clip.positions = burst_positions
pipeline.model.animator.add_animation_clip(burst_clip)

# 3. Export GLB containing 'escape_burst' alongside the default clips
custom_fish_output = output_dir / "quickstart_goldfish_custom_burst.glb"
pipeline.export(str(custom_fish_output))

print(
    f"[+] Successfully exported model with custom motion clip to: {custom_fish_output}"
)
print(
    f"[+] Final animation clips in asset: {[clip.name for clip in pipeline.model.animator.clips]}"
)

---
## 4. Model & Armature Inspection

You can inspect the generated bone hierarchies, DAG joints, and animation metadata directly in Python.

In [ ]:
armature = pipeline.model.armature
print(f"Total bones in armature: {len(armature.bones_list)}")
print("-" * 65)
print(f"{'Bone ID':<22} | {'Parent Bone':<18} | {'Length':<8}")
print("-" * 65)
for bone in armature.bones_list[:10]:
    parent_id = bone.parent.id if bone.parent else "None (Root)"
    print(f"{bone.id:<22} | {parent_id:<18} | {bone.length:.4f}")

print("\n" + "=" * 65)
print("Animation Tracks Summary:")
print("=" * 65)
for clip in pipeline.model.animator.clips:
    print(
        f" - Clip '{clip.name}': Duration={clip.duration:.2f}s, Keyframes={len(clip.positions)}, Loopable={clip.is_loopable}"
    )

---
## Conclusion & Next Steps

Now you can:
- Procedurally autorig and animate aquatic organisms with zero Blender dependency.
- Generate continuous travelling wave animations for serpentine creatures.
- Synthesize custom kinematics and multi-track animation clips using `generate_base_animation`.
- Export production-ready `.glb` assets compatible with Unity, Unreal Engine, Three.js, and WebGL.